In [ ]:
import nibabel as nb
import numpy as np
import pandas as pd
from collections import Counter

# Optional plotting imports
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False
    print("Warning: matplotlib not available. Visualizations will be skipped.")

def explore_cifti_atlas(atlas_path: str):
    """
    Comprehensive exploration of a CIFTI dlabel atlas file.
    
    Parameters:
    -----------
    atlas_path : str
        Path to the .dlabel.nii atlas file
    """
    
    print("=" * 80)
    print(f"EXPLORING ATLAS: {atlas_path.split('/')[-1]}")
    print("=" * 80)
    
    # Load the atlas
    try:
        atl = nb.load(atlas_path)
        print("✓ Atlas loaded successfully")
    except Exception as e:
        print(f"✗ Error loading atlas: {e}")
        return
    
    # 1. BASIC FILE INFORMATION
    print("\n" + "─" * 50)
    print("1. BASIC FILE INFORMATION")
    print("─" * 50)
    print(f"File path: {atlas_path}")
    print(f"File size: {get_file_size(atlas_path)}")
    print(f"Data type: {atl.get_fdata().dtype}")
    print(f"Nibabel image type: {type(atl)}")
    
    # 2. CIFTI HEADER INFORMATION
    print("\n" + "─" * 50)
    print("2. CIFTI HEADER INFORMATION")
    print("─" * 50)
    header = atl.header
    
    # Get matrix shape safely
    try:
        shape = atl.shape
        print(f"Matrix shape: {shape}")
        print(f"Number of dimensions: {len(shape)}")
        
        # Get axis information
        for i in range(len(shape)):
            try:
                axis = header.get_axis(i)
                print(f"Axis {i}: {type(axis).__name__} (size: {axis.size})")
            except Exception as e:
                print(f"Axis {i}: Could not retrieve axis info ({e})")
                
    except Exception as e:
        print(f"Could not retrieve matrix shape: {e}")
        # Try alternative method
        try:
            data = atl.get_fdata()
            print(f"Data shape from array: {data.shape}")
        except Exception as e2:
            print(f"Could not get shape: {e2}")
    
    # 3. GRAYORDINATE STRUCTURE (Axis 1)
    print("\n" + "─" * 50)
    print("3. GRAYORDINATE STRUCTURE")
    print("─" * 50)
    
    try:
        brain_axis = header.get_axis(1)  # Usually the spatial axis
        print(f"Brain axis type: {type(brain_axis).__name__}")
        print(f"Brain axis size: {brain_axis.size:,} grayordinates")
        
        if hasattr(brain_axis, 'iter_structures'):
            print("\nBrain structures found:")
            total_vertices = 0
            try:
                for i, structure in enumerate(brain_axis.iter_structures()):
                    # Handle different structure formats
                    if hasattr(structure, 'brain_structure'):
                        name = structure.brain_structure
                        indices = getattr(structure, 'vertex_indices', None)
                        count = len(indices) if indices is not None else getattr(structure, 'count', 'Unknown')
                    elif isinstance(structure, tuple) and len(structure) >= 2:
                        # Sometimes structures are returned as tuples
                        name = structure[0] if len(structure) > 0 else f"Structure_{i}"
                        count = structure[1] if len(structure) > 1 else "Unknown"
                    else:
                        name = f"Structure_{i}"
                        count = "Unknown"
                    
                    print(f"  • {name}: {count:,} vertices/voxels" if isinstance(count, int) else f"  • {name}: {count}")
                    if isinstance(count, int):
                        total_vertices += count
                        
                if total_vertices > 0:
                    print(f"Total grayordinates: {total_vertices:,}")
            except Exception as e:
                print(f"Could not iterate structures: {e}")
        else:
            print("No structure iteration method available")
            
    except Exception as e:
        print(f"Could not analyze grayordinate structure: {e}")
        try:
            # Fallback: just get the axis size
            brain_axis = header.get_axis(1)
            print(f"Grayordinate axis size: {brain_axis.size:,}")
        except:
            print("Could not determine grayordinate structure")
    
    # 4. LABEL INFORMATION (Axis 0) 
    print("\n" + "─" * 50)
    print("4. LABEL/PARCEL INFORMATION")
    print("─" * 50)
    
    label_axis = header.get_axis(0)
    if hasattr(label_axis, 'label_table'):
        label_table = label_axis.label_table
        print(f"Number of labels in table: {len(label_table)}")
        
        # Show label details
        print("\nLabel Table:")
        for i, (key, label_info) in enumerate(label_table.items()):
            if i < 20:  # Show first 20 labels
                rgba = getattr(label_info, 'rgba', [0, 0, 0, 0])
                print(f"  {key:3d}: {label_info.label:40s} RGBA: {rgba}")
            elif i == 20:
                print(f"  ... and {len(label_table) - 20} more labels")
                break
    
    # 5. DATA ARRAY ANALYSIS
    print("\n" + "─" * 50)
    print("5. DATA ARRAY ANALYSIS")
    print("─" * 50)
    
    # Load data
    data = atl.get_fdata()
    labels = np.squeeze(data).astype(int)
    
    print(f"Data shape: {data.shape}")
    print(f"Labels array shape: {labels.shape}")
    print(f"Data range: {labels.min()} to {labels.max()}")
    print(f"Unique labels: {len(np.unique(labels))}")
    print(f"Non-zero labels: {len(np.unique(labels[labels > 0]))}")
    
    # Label distribution
    unique_labels, counts = np.unique(labels, return_counts=True)
    
    print(f"\nLabel Distribution:")
    print(f"{'Label':<8} {'Count':<10} {'Percentage':<12} {'Description'}")
    print("-" * 60)
    
    # Get label names if available
    label_names = {}
    if hasattr(label_axis, 'label_table'):
        label_names = {k: v.label for k, v in label_axis.label_table.items()}
    
    for label, count in zip(unique_labels[:15], counts[:15]):  # Show first 15
        percentage = (count / len(labels)) * 100
        name = label_names.get(label, "Unknown")[:30]  # Truncate long names
        print(f"{label:<8} {count:<10,} {percentage:<11.2f}% {name}")
    
    if len(unique_labels) > 15:
        print(f"... and {len(unique_labels) - 15} more labels")
    
    # 6. NETWORK ANALYSIS (if network info available)
    print("\n" + "─" * 50)
    print("6. NETWORK ANALYSIS")
    print("─" * 50)
    
    analyze_networks(labels, label_names)
    
    # 7. SPATIAL COVERAGE
    print("\n" + "─" * 50)
    print("7. SPATIAL COVERAGE")
    print("─" * 50)
    
    background_count = np.sum(labels == 0)
    labeled_count = np.sum(labels > 0)
    total_grayordinates = len(labels)
    
    print(f"Total grayordinates: {total_grayordinates:,}")
    print(f"Background/unlabeled (label 0): {background_count:,} ({background_count/total_grayordinates*100:.1f}%)")
    print(f"Labeled regions: {labeled_count:,} ({labeled_count/total_grayordinates*100:.1f}%)")
    
    # 8. REGION SIZE STATISTICS
    print("\n" + "─" * 50)
    print("8. REGION SIZE STATISTICS")
    print("─" * 50)
    
    non_zero_labels = unique_labels[unique_labels > 0]
    non_zero_counts = counts[unique_labels > 0]
    
    if len(non_zero_counts) > 0:
        print(f"Number of brain regions: {len(non_zero_counts)}")
        print(f"Region size statistics:")
        print(f"  • Mean: {np.mean(non_zero_counts):.1f} vertices/voxels")
        print(f"  • Median: {np.median(non_zero_counts):.1f} vertices/voxels")
        print(f"  • Min: {np.min(non_zero_counts)} vertices/voxels")
        print(f"  • Max: {np.max(non_zero_counts)} vertices/voxels")
        print(f"  • Std: {np.std(non_zero_counts):.1f} vertices/voxels")
        
        # Find largest and smallest regions
        largest_idx = np.argmax(non_zero_counts)
        smallest_idx = np.argmin(non_zero_counts)
        largest_label = non_zero_labels[largest_idx]
        smallest_label = non_zero_labels[smallest_idx]
        
        print(f"\nLargest region:")
        print(f"  • Label {largest_label}: {label_names.get(largest_label, 'Unknown')} ({np.max(non_zero_counts)} vertices)")
        print(f"Smallest region:")
        print(f"  • Label {smallest_label}: {label_names.get(smallest_label, 'Unknown')} ({np.min(non_zero_counts)} vertices)")
    
    # 9. QUALITY CHECKS
    print("\n" + "─" * 50)
    print("9. QUALITY CHECKS")
    print("─" * 50)
    
    # Check for gaps in labeling
    expected_range = set(range(0, labels.max() + 1))
    actual_labels = set(unique_labels)
    missing_labels = expected_range - actual_labels
    
    if missing_labels:
        print(f"⚠️  Missing labels in sequence: {sorted(list(missing_labels))}")
    else:
        print("✓ No gaps in label sequence")
    
    # Check for very small regions
    tiny_regions = non_zero_counts[non_zero_counts < 10] if len(non_zero_counts) > 0 else []
    if len(tiny_regions) > 0:
        print(f"⚠️  {len(tiny_regions)} regions have fewer than 10 vertices/voxels")
    else:
        print("✓ All regions have reasonable size (≥10 vertices)")
    
    # 10. SUMMARY
    print("\n" + "=" * 50)
    print("ATLAS SUMMARY")
    print("=" * 50)
    print(f"Atlas name: {atlas_path.split('/')[-1]}")
    print(f"Total grayordinates: {total_grayordinates:,}")
    print(f"Number of brain regions: {len(non_zero_labels) if len(non_zero_labels) > 0 else 0}")
    print(f"Coverage: {labeled_count/total_grayordinates*100:.1f}% of brain")
    if len(non_zero_counts) > 0:
        print(f"Average region size: {np.mean(non_zero_counts):.1f} vertices/voxels")
    
    return {
        'atlas_path': atlas_path,
        'total_grayordinates': total_grayordinates,
        'num_regions': len(non_zero_labels) if len(non_zero_labels) > 0 else 0,
        'coverage_percent': labeled_count/total_grayordinates*100,
        'labels': labels,
        'label_names': label_names,
        'unique_labels': unique_labels,
        'counts': counts
    }


def analyze_networks(labels, label_names):
    """Analyze network structure if network info is in label names"""
    
    # Look for network keywords in label names
    network_keywords = ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']
    
    network_counts = {}
    for label_id, name in label_names.items():
        if label_id == 0:  # Skip background
            continue
            
        # Find which network this region belongs to
        found_network = None
        for network in network_keywords:
            if network.lower() in name.lower():
                found_network = network
                break
        
        if found_network:
            if found_network not in network_counts:
                network_counts[found_network] = 0
            network_counts[found_network] += 1
    
    if network_counts:
        print("Network composition:")
        for network, count in sorted(network_counts.items()):
            print(f"  • {network}: {count} regions")
    else:
        print("No clear network structure detected in label names")


def get_file_size(filepath):
    """Get human readable file size"""
    import os
    size = os.path.getsize(filepath)
    for unit in ['B', 'KB', 'MB', 'GB']:
        if size < 1024.0:
            return f"{size:.1f} {unit}"
        size /= 1024.0
    return f"{size:.1f} TB"


def create_atlas_visualization(atlas_info):
    """Create visualizations of the atlas"""
    
    if not PLOTTING_AVAILABLE:
        print("Matplotlib not available. Skipping visualizations.")
        print("To enable plots, install matplotlib: pip install matplotlib")
        return
    
    labels = atlas_info['labels']
    unique_labels = atlas_info['unique_labels']
    counts = atlas_info['counts']
    
    # Remove background for plotting
    non_zero_mask = unique_labels > 0
    plot_labels = unique_labels[non_zero_mask]
    plot_counts = counts[non_zero_mask]
    
    if len(plot_counts) == 0:
        print("No non-zero labels to plot")
        return
    
    try:
        # Create figure with subplots
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Region size histogram
        ax1.hist(plot_counts, bins=30, alpha=0.7, edgecolor='black')
        ax1.set_xlabel('Region Size (vertices/voxels)')
        ax1.set_ylabel('Number of Regions')
        ax1.set_title('Distribution of Region Sizes')
        ax1.grid(True, alpha=0.3)
        
        # 2. Cumulative size plot
        sorted_counts = np.sort(plot_counts)[::-1]
        cumsum = np.cumsum(sorted_counts)
        ax2.plot(range(1, len(cumsum) + 1), cumsum / cumsum[-1] * 100)
        ax2.set_xlabel('Region Rank')
        ax2.set_ylabel('Cumulative Coverage (%)')
        ax2.set_title('Cumulative Region Coverage')
        ax2.grid(True, alpha=0.3)
        
        # 3. Top 20 largest regions
        if len(plot_counts) >= 20:
            top_20_idx = np.argsort(plot_counts)[-20:]
            top_20_labels = plot_labels[top_20_idx]
            top_20_counts = plot_counts[top_20_idx]
            
            ax3.barh(range(20), top_20_counts)
            ax3.set_yticks(range(20))
            ax3.set_yticklabels([f"Region {label}" for label in top_20_labels])
            ax3.set_xlabel('Size (vertices/voxels)')
            ax3.set_title('Top 20 Largest Regions')
        else:
            ax3.text(0.5, 0.5, f'Only {len(plot_counts)} regions\nfound in atlas', 
                    ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title('Region Sizes')
        
        # 4. Label distribution overview
        coverage_data = [
            np.sum(labels == 0),  # Background
            np.sum(labels > 0)    # All regions
        ]
        ax4.pie(coverage_data, labels=['Background', 'Brain Regions'], autopct='%1.1f%%', startangle=90)
        ax4.set_title('Brain Coverage')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error creating visualizations: {e}")
        print("Atlas exploration completed successfully, but plots failed.")


# Example usage
# if __name__ == "__main__":
#     # Update this path to your atlas file
#     atlas_path = "Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii"
    
#     # Explore the atlas
#     atlas_info = explore_cifti_atlas(atlas_path)
    
#     # Create visualizations
#     if atlas_info:
#         print("\nCreating visualizations...")
#         create_atlas_visualization(atlas_info)

In [ ]:
atlas_info = explore_cifti_atlas("data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii")


In [ ]:
# Run exploration and get results
atlas_info = explore_cifti_atlas("data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2.dlabel.nii")

# Create visualizations
if atlas_info:
    create_atlas_visualization(atlas_info)